## Packages and Libraries

In [ ]:
#Análisis Exploratorio Calidad de Aire en Castilla
# STP
# 25/02/24

import numpy as np # Library for mathematical arrays operations
import pandas as pd # Library for manipulation data and analysis
import matplotlib.pyplot as plt # Library for plotting
import seaborn as sns #Used for remove outliers in boxplot
plt.style.use('ggplot')
%matplotlib inline

## Data Exploratory Analysis

In [ ]:
df = pd.read_csv('calidad.csv',sep = ";") #Once the file is uploaded, df is the name of the .csv file
df  #Show the dataframe

In [ ]:
print(df.info())  #Show the type of data that is df

In [ ]:
print(df.head())  #Show the 5 initial data for each attribute

In [ ]:
print(df.describe())  #Statistical summary and description
#Categorical variables are not taken into account

In [ ]:
MeanDF=df['CO (mg/m3)'].mean()
MeanDF

In [ ]:
#Renaming column names for plotting
df1 = df.rename(columns={'CO (mg/m3)': 'CO','NO (ug/m3)': 'NO','NO2 (ug/m3)': 'NO2',
                         'O3 (ug/m3)': 'O3','PM10 (ug/m3)': 'PM10','PM25 (ug/m3)': 'PM25','SO2 (ug/m3)': 'SO2'})
df1

In [ ]:
#Plotting histograms
#For 03
plt.figure(figsize=(10, 5)) #Figure size
plt.hist(df1['O3'], bins=1000, color='steelblue', alpha=0.7)
plt.title('Histograma de O3')
plt.xlabel('O3 (ug/m3)')
plt.ylabel('Frecuencia')
plt.xlim(0, 150)

plt.show()

In [ ]:
#For PM10

plt.figure(figsize=(10, 5))
plt.hist(df1['PM10'], bins=1000, color='steelblue', alpha=0.7)
plt.title('Histograma de PM10')
plt.xlabel('PM10 (ug/m3)')
plt.ylabel('Frecuencia')
plt.xlim(0, 150)

plt.show()

## Setting variable types

In [ ]:
#Change the date format
df1['Fecha'] = pd.to_datetime(df1['Fecha'], format='%Y/%m/%d').dt.strftime('%d/%m/%Y')
df1['Fecha']

In [ ]:
#Show the unique values in 'Provincia' atribute
Prov = df1['Provincia'].unique()
print(Prov)

In [ ]:
#Show the unique values in 'Estacion' atribute
Est = df1['Estación'].unique()
print(Est)

## Detection and Treatment of Lost Data

In [ ]:
#Identify if exists NANs values in the columns
NANs_List = df1.isna()
print(NANs_List)

In [ ]:
#Identify if exists at least one NANs values in the columns
NANs = df1.isna().any()#.any() #other any.() to show at least if the df contains empty data
print(NANs)

In [ ]:
#Sums the NANs values in the df or by atributes
NANs_Sum = df1.isna().sum()#.sum() #other sum.() to show the total NAs in df
print(NANs_Sum)

In [ ]:
#Calculate the mean of NaNs in each atribute or in df
NANs_Mean = df1.isna().mean()#.mean() #other sum.() to show the total mean of NANs in df
print(NANs_Mean)

In [ ]:
#Keeping the columns with less than 50% of mean
colMeans = df1.columns[df1.isna().mean() < 0.50]
df2 = df1[colMeans]
df2 #This will eliminate CO and PM25

In [ ]:
#Calculate mean in df2 rounding with 2 decimals
NANs_df2 = df2.isna().mean().round(2)
print(NANs_df2)

In [ ]:
#Replacing NANs with mean
NumCol = df2.select_dtypes(include='number').columns
MeanCol = df2[NumCol].mean(skipna=True)  #na_rm ignores the NANs values in df2

for i in NumCol:
    df2[i].fillna(MeanCol[i], inplace=True) #i iterates over df2 and fill the NANs with the df mean. For numeric values

In [ ]:
#Verify if still exists NANs in df
NANs_after = df2.isna().sum()
print(NANs_after)

In [ ]:
df2.round(5)

## Analysing Outliers

In [ ]:
#Analog to R plotting setting 'histograma' as main variable with features
plt.figure(figsize=(5, 5))
hist_data, bin_edges, _  = plt.hist(df1['O3'], bins=1000, color='red', alpha=0.5)
plt.title('Histograma de O3')
plt.xlabel('O3 (ug/m3)')
plt.ylabel('Frecuencia')
plt.xlim(0, 150)
plt.grid(True)
plt.show()

#Histogram details
#print("Histogram bars data:")
#print(hist_data)
#print("\nHistogram bins limits:")
#print(bin_edges)


In [ ]:
outliers = plt.boxplot(df2['O3'], vert=False) #If there's NAN values, doesn't plot
plt.xlabel('O3 (ug/m3)')
plt.title('Boxplot de O3')
plt.show()

In [ ]:
#Boxplot stats for O3
print("Estadísticas del boxplot:")
print(outliers)

In [ ]:
#Eliminating outliers from boxplot with seaborn library
def removal_box_plot(df, column, threshold1, threshold2):
    sns.boxplot(df[column])
    plt.title(f'Original Box Plot of {column}')
    plt.show()

    rm = df[(df[column] >= threshold1) & (df[column] <= threshold2)]

    sns.boxplot(rm['O3'])
    plt.title(f'Box Plot without Outliers of {column}')
    plt.show()
    return rm

threshold_value1 = 29
threshold_value2 = 49

no_outliers = removal_box_plot(df2, 'O3', threshold_value1,threshold_value2)

In [ ]:
print(no_outliers.info())
print(no_outliers.describe())

In [ ]:
print(df1.count())

In [ ]:
NumCategoria = df1['Provincia'].value_counts()
print(NumCategoria)

In [ ]:
#Plotting an bar diagram
plt.figure(figsize=(10, 6))
df1['Provincia'].value_counts().plot(kind='bar', color='skyblue')
plt.xlabel('Provincias')
plt.ylabel('Nº observaciones')
plt.title('Distribución de la variable Provincia')
plt.xticks(rotation=30, ha='right')  #Rotates the x labels
plt.show()

In [ ]:
#Removing categoric outliers
df3 = df2[df2['Provincia'] != 'Madrid'] #Eliminates 'Madrid' from 'Provincia' column
df3.loc[:,"Provincia"].unique() #Show the column 'Provincia' and its unique values

In [ ]:
#Replotting the 'Provincia' atribute
plt.figure(figsize=(10, 6))
df3['Provincia'].value_counts().plot(kind='bar', color='skyblue')
plt.xlabel('Provincias')
plt.ylabel('Nº observaciones')
plt.title('Distribución de la variable Provincia')
plt.xticks(rotation=30, ha='right')  #Rotates the x labels
plt.show()

## Correlation Analysis

In [ ]:
#Selecting the numerical variables
NVariables = df1.select_dtypes(include='number')
NVariables

In [ ]:
#Developing the correlation matrix for numerical values in df
correlation = NVariables.corr()
correlation

In [ ]:
#Plotting the correlation matrix
plt.figure(figsize=(8, 7))
sns.heatmap(correlation, annot=True, cmap='coolwarm', fmt=".2f")
plt.title('Matriz de Correlaciones')
plt.show()
